# Customer Data Enrichment

This notebook restores the Customer_ID column from the cleaned customer dataset and integrates it into the final customer churn and LTV datasets.

The Customer_ID column was removed during machine learning preprocessing because it does not contribute to churn prediction. However, for business reporting, customer retention activities, dashboard development, and PostgreSQL analysis, customer identifiers are required to uniquely identify customers.

The enrichment process ensures that each customer record in the final analytics dataset can be traced back to its original customer identifier while keeping the machine learning pipeline unchanged.

Key activities performed:

* Loaded the cleaned customer dataset containing Customer_ID.
* Verified row consistency between cleaned and final datasets.
* Restored Customer_ID to the final LTV dataset.
* Regenerated the high-priority customer dataset with Customer_ID included.
* Prepared enriched datasets for PostgreSQL storage and business reporting.


In [2]:
import pandas as pd

In [4]:
clean_df = pd.read_csv("../data/processed/cleaned_telco_data.csv")

clean_df.shape

(7032, 21)

In [5]:
clean_df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

In [6]:
ltv_df = pd.read_csv(
    "../data/processed/customer_churn_ltv_final.csv"
)

print(clean_df.shape)
print(ltv_df.shape)

(7032, 21)
(7032, 34)


In [7]:
clean_df[["customerID", "MonthlyCharges"]].head()

,customerID,MonthlyCharges
0,7590-VHVEG,29.85
1,5575-GNVDE,56.95
2,3668-QPYBK,53.85
3,7795-CFOCW,42.30
4,9237-HQITU,70.70


In [8]:
ltv_df[["MonthlyCharges"]].head()

,MonthlyCharges
0,29.85
1,56.95
2,53.85
3,42.30
4,70.70


In [9]:
ltv_df.insert(
    0,
    "Customer_ID",
    clean_df["customerID"]
)

In [10]:
ltv_df.head()

,Customer_ID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Estimated_LTV,LTV_Segment,Customer_Priority
0,7590-VHVEG,1,0,1,0,1,0,1,29.85,29.85,...,0,0,0,0,0,1,0,29.85,Low Value,Low Value - Low Risk
1,5575-GNVDE,0,0,0,0,34,1,0,56.95,1889.50,...,0,0,1,0,0,0,1,1936.30,Medium Value,Medium Value - Low Risk
2,3668-QPYBK,0,0,0,0,2,1,1,53.85,108.15,...,0,0,0,0,0,0,1,107.70,Low Value,Low Value - High Risk
3,7795-CFOCW,0,0,0,0,45,0,0,42.30,1840.75,...,0,0,1,0,0,0,0,1903.50,Medium Value,Medium Value - Low Risk
4,9237-HQITU,1,0,0,0,2,1,1,70.70,151.65,...,0,0,0,0,0,1,0,141.40,Low Value,Low Value - High Risk


In [11]:
ltv_df.to_csv(
    "../data/processed/customer_churn_ltv_final.csv",
    index=False
)

In [12]:
high_priority_customers = ltv_df[
    ltv_df["Customer_Priority"] == "High Value - High Risk"
]

In [13]:
high_priority_customers.to_csv(
    "../data/processed/high_priority_customers.csv",
    index=False
)

print("High priority customers CSV updated successfully.")

High priority customers CSV updated successfully.


In [15]:
pd.read_csv("../data/processed/high_priority_customers.csv").head(2)

,Customer_ID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Estimated_LTV,LTV_Segment,Customer_Priority
0,0280-XJGEX,0,0,0,0,49,1,1,103.70,5036.30,...,0,1,0,0,0,0,0,5081.30,High Value,High Value - High Risk
1,6467-CHFZW,0,0,1,1,47,1,1,99.35,4749.15,...,0,1,0,0,0,1,0,4669.45,High Value,High Value - High Risk
